In [31]:
from numpy.random import default_rng
import numpy as np
###################################

from pathlib import Path
import sys

In [32]:
from src.simuladorBJPC import BJPCPlan, simulate_bjpc_exp_suffstats
from src.priors import GammaPrior, BetaPrior, BetaGammaPrior
from src.loss import WarrantyTargets, AcceptanceWeights, g_exp_fail_before_L

In [33]:
############################### DATA BJPC #####################################
plan = BJPCPlan(n=7, k=6, R=[1,0,0,0,0])
rng = default_rng(123)

data = simulate_bjpc_exp_suffstats(plan, lambda1=0.06, lambda2=0.03, rng=rng)
data

{'k1': 4, 'k2': 2, 'u': 73.51248704224477, 't_end': 30.2922991916515}

In [22]:
################################## PRIOR #####################################
gamma_prior = GammaPrior(a0=2.0, b0=1.5)
beta_prior  = BetaPrior(a1=2.0, a2=3.0)

bg_prior = BetaGammaPrior(gamma_prior=gamma_prior, beta_prior=beta_prior)

lambda1, lambda2 = bg_prior.sample(size=2)
print(lambda1, lambda2)

[0.48930019 0.1513659 ] [1.66256778 0.85601333]


In [23]:
############################### POSTERIOR #####################################
post = BetaGammaPosterior(
    a0=2.0, b0=1.0,
    a1=1.0, a2=1.0,
    k1=7, k2=3, u=120.5
)

# Muestrear lambda1, lambda2 
rng = default_rng(123)
lambda1_s, lambda2_s = post.sample(size=10000, rng=rng)

print(lambda1_s.mean(), lambda2_s.mean())

0.06591128395022802 0.03313578862001775


In [26]:
############################### LOSS #####################################
targets = WarrantyTargets(L1=100.0, L2=100.0)     # vida/garantía
weights = AcceptanceWeights(C1=200.0, C2=200.0)   # costo por falla temprana

#cálculo de g para cada posterior
g_s = g_exp_fail_before_L(
    lambda1=lambda1_s,
    lambda2=lambda2_s,
    targets=targets,
    weights=weights
)

phi = float(g_s.mean())
phi


380.3686329019304

In [ ]:
############################### DECISION #####################################
"""
Aceptas si 𝜙 < 𝐶𝑟
Rechazas si 𝜙 ≥ 𝐶𝑟
"""

Cr = 150.0
accept = phi < Cr
print("phi =", phi, "| decision =", "ACCEPT" if accept else "REJECT")

phi = 380.3686329019304 | decision = REJECT


In [ ]:
############################### RIESGO DE BAYES #####################################
